# 04 — Reconstruct Redacted PDFs
Reads per-page `.json` files from `redacted_text/` and writes PDFs to `output_folder/`.

Each source document produces three output files:
- `redacted_{stem}.pdf` — sanitized content only (one page per original page)
- `summary_{stem}.pdf` — standalone PII/PHI replacement mapping table
- `governance_{stem}.json` — machine-readable audit log with per-page redaction details

In [ ]:
%pip install pymupdf boto3 Pillow reportlab --prefer-binary -q

In [ ]:
import json, os, sys
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER

BASE_DIR      = Path(os.getcwd()).parent
sys.path.insert(0, str(BASE_DIR))

from utils import get_logger
from models import resolve_model_id, get_audit_model_id
logger = get_logger("04_reconstruct_pdf")

TEXT_FOLDER   = BASE_DIR / "redacted_text"
OUTPUT_FOLDER = BASE_DIR / "output_folder"

OUTPUT_FOLDER.mkdir(exist_ok=True)
logger.info("Text folder  : %s", TEXT_FOLDER)
logger.info("Output folder: %s", OUTPUT_FOLDER)

In [ ]:
def load_pages(pdf_stem: str) -> tuple[list[str], list[dict], list[list[dict]]]:
    """
    Load all page JSON files for a given PDF stem.
    Returns (page_texts, deduplicated_mapping_rows, per_page_mappings).
    """
    page_files = sorted(
        TEXT_FOLDER.glob(f"{pdf_stem}_page_*.json"),
        key=lambda p: int(p.stem.rsplit("_page_", 1)[-1])
    )
    logger.debug("Found %d page file(s) for %s", len(page_files), pdf_stem)

    page_texts: list[str] = []
    page_mappings: list[list[dict]] = []
    all_mapping: list[dict] = []
    seen: set[tuple] = set()

    for pf in page_files:
        data = json.loads(pf.read_text(encoding="utf-8"))
        page_texts.append(data.get("sanitized_text", ""))
        page_mapping = data.get("mapping", [])
        page_mappings.append(page_mapping)
        for row in page_mapping:
            key = (row["original_masked"], row["type"])
            if key not in seen:
                all_mapping.append(row)
                seen.add(key)

    logger.debug("%s: %d pages, %d unique mapping entries",
                 pdf_stem, len(page_texts), len(all_mapping))
    return page_texts, all_mapping, page_mappings


def xml_escape(text: str) -> str:
    return (
        text
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


def build_redacted_pdf(pdf_stem: str, page_texts: list[str], out_path: Path) -> None:
    """Write sanitized content pages only — no summary table."""
    logger.info("Building redacted PDF: %s (%d pages)", out_path.name, len(page_texts))

    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    body = ParagraphStyle(
        "Body", parent=styles["Normal"],
        fontName="Courier", fontSize=9, leading=13, alignment=TA_LEFT,
    )
    story = []
    for page_idx, text in enumerate(page_texts):
        if page_idx > 0:
            story.append(PageBreak())
        logger.debug("  Rendering page %d (%d chars)", page_idx + 1, len(text))
        for line in text.splitlines():
            safe = xml_escape(line)
            story.append(Paragraph(safe, body) if safe.strip() else Spacer(1, 6))

    doc.build(story)
    logger.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


def build_summary_pdf(pdf_stem: str, mapping: list[dict], out_path: Path) -> None:
    """Write a standalone summary PDF with the PII/PHI replacement table."""
    logger.info("Building summary PDF: %s (%d entries)", out_path.name, len(mapping))

    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    heading = ParagraphStyle(
        "Heading", parent=styles["Heading2"],
        alignment=TA_CENTER, spaceAfter=12,
    )
    story = [
        Paragraph(f"PII / PHI Replacement Summary — {pdf_stem}", heading),
        Spacer(1, 6),
    ]

    table_data = [["Original (masked)", "Replacement", "Data Type"]]
    for row in mapping:
        table_data.append([
            row.get("original_masked", ""),
            row.get("replacement", ""),
            row.get("type", ""),
        ])

    tbl = Table(table_data, colWidths=[6*cm, 7*cm, 4*cm], repeatRows=1)
    tbl.setStyle(TableStyle([
        ("BACKGROUND",    (0, 0), (-1, 0),  colors.HexColor("#2E4057")),
        ("TEXTCOLOR",     (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",      (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS",(0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F2F2")]),
        ("GRID",          (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN",        (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING",   (0, 0), (-1, -1), 4),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 4),
        ("TOPPADDING",    (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]))
    story.append(tbl)
    doc.build(story)
    logger.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


def build_governance_json(pdf_stem: str, source_name: str, page_mappings: list,
                          model_id: str, audit_model_id: str, out_path: Path) -> None:
    """Write a governance/audit JSON log for the redacted document."""
    all_redactions = sum(len(pm) for pm in page_mappings)
    categories = sorted({r["type"] for pm in page_mappings for r in pm})

    pages = []
    for i, pm in enumerate(page_mappings, 1):
        pages.append({"page_number": i, "redactions": pm})

    # Consolidated mapping with page references
    seen = {}
    for i, pm in enumerate(page_mappings, 1):
        for r in pm:
            key = (r["original_masked"], r["type"])
            if key not in seen:
                seen[key] = {**r, "pages": [i]}
            elif i not in seen[key]["pages"]:
                seen[key]["pages"].append(i)

    gov = {
        "source_file": source_name,
        "processed_at": datetime.now(tz=timezone.utc).isoformat(timespec="seconds"),
        "model_id": model_id,
        "audit_model_id": audit_model_id,
        "total_pages": len(page_mappings),
        "total_redactions": all_redactions,
        "categories_found": categories,
        "pages": pages,
        "consolidated_mapping": list(seen.values()),
    }
    out_path.write_text(json.dumps(gov, ensure_ascii=False, indent=2), encoding="utf-8")
    logger.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


logger.info("Functions defined")

In [ ]:
# Discover all PDF stems from available .json files
json_files = sorted(TEXT_FOLDER.glob("*.json"))
logger.info("Found %d page file(s) in redacted_text/", len(json_files))

stems: set[str] = set()
for jf in json_files:
    stems.add(jf.stem.rsplit("_page_", 1)[0])

for s in sorted(stems):
    count = len(list(TEXT_FOLDER.glob(f"{s}_page_*.json")))
    logger.info("  %s: %d page(s)", s, count)

In [ ]:
# Build one redacted PDF + one summary PDF + one governance JSON per source document
model_id = resolve_model_id(None)
audit_model_id = get_audit_model_id()

for pdf_stem in sorted(stems):
    page_texts, mapping, page_mappings = load_pages(pdf_stem)

    redacted_path   = OUTPUT_FOLDER / f"redacted_{pdf_stem}.pdf"
    summary_path    = OUTPUT_FOLDER / f"summary_{pdf_stem}.pdf"
    governance_path = OUTPUT_FOLDER / f"governance_{pdf_stem}.json"

    build_redacted_pdf(pdf_stem, page_texts, redacted_path)
    build_summary_pdf(pdf_stem, mapping, summary_path)
    build_governance_json(pdf_stem, f"{pdf_stem}.pdf", page_mappings,
                          model_id, audit_model_id, governance_path)

logger.info("All outputs written to output_folder/")

In [ ]:
# Verify outputs
for p in sorted(OUTPUT_FOLDER.glob("redacted_*.pdf")):
    logger.info("  %s  (%.1f KB)", p.name, p.stat().st_size / 1024)
for p in sorted(OUTPUT_FOLDER.glob("governance_*.json")):
    logger.info("  %s  (%.1f KB)", p.name, p.stat().st_size / 1024)